# 📘 Databricks Delta Lake — Detailed Notes (SQL + PySpark + Deep Explanations)

---

## 🔷 1. What is Delta Lake?

👉 Delta Lake = **Data Lake + Reliability Layer**

- Stores data as **Parquet files**
- Maintains **_delta_log** for tracking changes

---

### 🧠 Mental Model

```
Data Files (Parquet)
        +
_delta_log (Transactions)
        =
Delta Table
```

---

## 🔷 2. Create Delta Table

### SQL
```sql
CREATE TABLE sales.analytics.customers USING DELTA;
```

### PySpark
```python
df.write.format("delta").saveAsTable("sales.analytics.customers")
```

---

## 🔷 3. Insert / Update / Delete / Merge

### INSERT

```sql
INSERT INTO customers SELECT * FROM source;
```

```python
df.write.mode("append").saveAsTable("customers")
```

---

### UPDATE

```sql
UPDATE customers SET age = 30 WHERE id = 1;
```

```python
from delta.tables import DeltaTable

delta = DeltaTable.forName(spark, "customers")
delta.update("id = 1", {"age": "30"})
```

---

### DELETE

```sql
DELETE FROM customers WHERE id = 1;
```

```python
delta.delete("id = 1")
```

---

### MERGE (UPSERT)

👉 Most important operation in real-world

---

### SQL
```sql
MERGE INTO customers t
USING updates s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

---

### PySpark
```python
delta.alias("t").merge(
  updates.alias("s"),
  "t.id = s.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
```

---

### 🧠 Why MERGE?

👉 Used in:
- CDC pipelines
- Incremental updates
- Slowly Changing Dimensions (SCD)

---

## 🔷 4. Time Travel

---

### SQL
```sql
SELECT * FROM customers VERSION AS OF 1;
```

---

### PySpark
```python
spark.read.format("delta") \
  .option("versionAsOf", 1) \
  .load("/data")
```

---

### 🧠 Why Time Travel?

- Debug issues
- Recover deleted data
- Audit history

---

## 🔷 5. Schema Evolution

---

### SQL
```sql
ALTER TABLE customers ADD COLUMNS (new_col STRING);
```

---

### PySpark
```python
df.write.format("delta") \
  .option("mergeSchema", "true") \
  .mode("append") \
  .save("/data")
```

---

### 🧠 Why Needed?

👉 Real data keeps changing (new columns)

---

## 🔷 6. COPY INTO

---

### SQL
```sql
COPY INTO customers
FROM '/data'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');
```

---

### 🧠 Why Use?

- Incremental file ingestion
- Avoid duplicate loads

---

## 🔷 7. CTAS (Create Table As Select)

---

### SQL
```sql
CREATE TABLE new_table
USING DELTA
AS SELECT * FROM customers;
```

---

### PySpark
```python
spark.sql("CREATE TABLE new_table USING DELTA AS SELECT * FROM customers")
```

---

### 🧠 Why Use?

- Quick table creation
- Data transformation + creation in one step

---

## 🔷 8. OPTIMIZE (Very Important 🔥)

---

### Problem:

👉 Too many small files

---

### Why small files are bad?

- Slow queries
- More file scans
- High metadata overhead

---

### Solution:

```sql
OPTIMIZE customers;
```

```python
spark.sql("OPTIMIZE customers")
```

---

### 🧠 What OPTIMIZE does?

- Combines small files → large files
- Improves read performance

---

## 🔷 9. Z-ORDER (Advanced Optimization 🔥🔥)

---

### Problem:

👉 Even after optimize, queries still scan many files

---

### Solution:

```sql
OPTIMIZE customers ZORDER BY (id);
```

---

### 🧠 What Z-ORDER does?

👉 Reorganizes data based on column values

---

### Example:

Query:
```sql
SELECT * FROM customers WHERE id = 100;
```

Without ZORDER:
- Scans many files ❌

With ZORDER:
- Reads only relevant files ✅

---

### 🧠 Use ZORDER when:

- Frequently filter on column
- Large dataset

---

## 🔷 10. VACUUM (Cleanup)

---

### SQL
```sql
VACUUM customers RETAIN 168 HOURS;
```

---

### 🧠 What it does?

- Deletes old unused files

---

### Why needed?

- Save storage
- Remove stale data

---

⚠️ Important:
- Keep retention ≥ 7 days (for time travel safety)

---

## 🔷 11. Change Data Feed (CDF)

---

### Enable

```sql
ALTER TABLE customers
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
```

---

### Read Changes

```sql
SELECT * FROM table_changes('customers', 1);
```

---

### 🧠 Why CDF?

- Track inserts, updates, deletes
- Build incremental pipelines

---

## 🔷 12. Streaming with Delta

---

### PySpark
```python
df = spark.readStream.table("customers")

df.writeStream \
  .option("checkpointLocation", "/chk") \
  .toTable("output")
```

---

### 🧠 Why Delta in Streaming?

- Exactly-once guarantee
- Fault tolerance

---

## 🔷 13. Real-World Scenario

---

### Example: Daily Incremental Load

1. Load new data
2. Merge into target
3. Optimize
4. Vacuum

---

### Code

```python
df = spark.read.format("csv").load("/input")

from delta.tables import DeltaTable
delta = DeltaTable.forName(spark, "customers")

delta.alias("t").merge(
  df.alias("s"),
  "t.id = s.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
```

---

## 🔷 14. Final Mental Model

```
Ingest → Store (Delta) → Update (MERGE) → Optimize → Query → Maintain
```

---

## 🔷 🚀 One-Line Summary

> Delta Lake = Reliable + Optimized + Versioned storage with SQL + PySpark support
